### Points to fix in Processing (proposed by Claude)

- Year gaps: income and GDP are NaN for 2013
    - Decision: leave NaN
- Kfz_je_1000_EW exists only for 2013/2017 and Pkw_je_1000_EW only for 2021/2025, and they are not the same measure.
    - Decision: drop both columns (cant be combined bc not defined same, in our dashboard we would have a drop from about 669 to 578 bc of definition break. 
- Definition break: Schulabgaenger_Abitur_Pct includes Fachhochschulreife from 2017
    - add a second, new column that only says which definition each row's number uses:
        struktur["Jahr"] >= 2017 compares each row's year with 2017 and gives True or False per row.
        struktur["Abitur_inkl_Fachhochschulreife"] = ... stores those True/False values as a new column.
- Small gaps: Beschaeftigte_Produzierendes_Gewerbe_Pct has 6 NaN.
    - Decision: keep them, 
- Footnotes: Fussnote (20% of rows) marks city-wide values repeated across constituencies; flag these for the regression.
- Boundaries: 317 distinct names for 299 numbers, so constituencies were renamed or redrawn between elections. Compare within a year, not as a panel, unless you map them.
    - The processing script's docstring: one line saying "Join on year + Wahlkreis number; numbers are only valid within one election year."

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
struktur = pd.read_csv('../data/interim/btw_strukturdaten.csv')

### Year Gaps for Income and GDP


In [ ]:
# Confirmation of Claudes proposal: 

struktur.loc[(struktur['Jahr'] == 2013) & (struktur['Verfuegbares_Einkommen_EUR_je_EW'].notna())]['Verfuegbares_Einkommen_EUR_je_EW']

struktur.loc[(struktur['Jahr'] == 2013) & (struktur['BIP_EUR_je_EW'].notna())]['BIP_EUR_je_EW']

# Both Columns return nothing for not na entries, Claudes proposal is correct and its because these data were not published. 
# We could fill up the data from other sources but decide not to right now and to leave them NaN.

Series([], Name: BIP_EUR_je_EW, dtype: float64)

In [ ]:
# Do income and GDP add information about the AfD share beyond the other structural variables?
# Method: fit a linear regression (OLS) of the AfD Zweitstimmen share on all other structural variables
# plus an East/West flag, then add income and GDP and compare R² (share of variance explained).
# If R² barely rises, income and GDP carry no extra information: the other variables already cover it.
struktur = pd.read_csv("../data/interim/btw_strukturdaten.csv")
wk = pd.read_csv("../data/interim/btw_wahlkreise.csv")
wk["AfD"] = wk["Zweitstimmen AfD"] / wk["Gültige Zweitstimmen"] * 100
merged = struktur.merge(wk[["Jahr", "Nr", "AfD"]], left_on=["Jahr", "Wahlkreis_Nr"], right_on=["Jahr", "Nr"])

# East/West flag: 1 for the five eastern states (Berlin counted as West, it is split East/West itself)
ost_laender = ["Brandenburg", "Mecklenburg-Vorpommern", "Sachsen", "Sachsen-Anhalt", "Thüringen"]
merged["Ost"] = merged["Land"].isin(ost_laender).astype(float)

income_gdp = ["Verfuegbares_Einkommen_EUR_je_EW", "BIP_EUR_je_EW"]
# Base model: every numeric structural variable except IDs, income/GDP, and Kfz/Pkw
# (Kfz/Pkw are split across two columns by year, so each year has one of them empty)
exclude = ["Jahr", "Wahlkreis_Nr", "Nr", "AfD", "Kfz_je_1000_EW", "Pkw_je_1000_EW"] + income_gdp
base_vars = [c for c in merged.columns if pd.api.types.is_numeric_dtype(merged[c]) and c not in exclude]


def r_squared(data, predictors, target="AfD"):
    """R² of an OLS fit target ~ predictors (with intercept), via numpy least squares."""
    X = np.column_stack([np.ones(len(data)), data[predictors].values])
    y = data[target].values
    coef, *_ = np.linalg.lstsq(X, y, rcond=None)
    residuals = y - X @ coef
    return 1 - residuals.var() / y.var()


rows = []
for year, g in merged[merged["Jahr"] > 2013].groupby("Jahr"):   # 2013 has no income/GDP
    g = g.dropna(subset=base_vars + income_gdp + ["AfD"])        # OLS needs complete rows
    r2_base = r_squared(g, base_vars)
    r2_full = r_squared(g, base_vars + income_gdp)
    rows.append({"Jahr": year, "Wahlkreise": len(g), "R² ohne Einkommen/BIP": round(r2_base, 3),
                 "R² mit Einkommen/BIP": round(r2_full, 3), "Zuwachs": round(r2_full - r2_base, 3)})
r2_table = pd.DataFrame(rows).set_index("Jahr")
display(r2_table)

# Why the gain is small: the base variables already predict income and GDP themselves.
# R² of income ~ base variables and GDP ~ base variables, plus the base variables they correlate with most.
overlap_rows = []
for year, g in merged[merged["Jahr"] > 2013].groupby("Jahr"):
    g = g.dropna(subset=base_vars + income_gdp)
    for var in income_gdp:
        top = g[base_vars].corrwith(g[var]).sort_values(key=abs, ascending=False).head(3).round(2)
        overlap_rows.append({"Jahr": year, "Variable": var,
                             "R² aus Basisvariablen": round(r_squared(g, base_vars, target=var), 2),
                             "stärkste Korrelationen": ", ".join(f"{k} ({v})" for k, v in top.items())})
overlap = pd.DataFrame(overlap_rows).set_index(["Jahr", "Variable"])
display(overlap)
print("Base model variables:", base_vars)

,Wahlkreise,R² ohne Einkommen/BIP,R² mit Einkommen/BIP,Zuwachs
Jahr,,,,
2017,293,0.773,0.776,0.003
2021,299,0.852,0.854,0.002
2025,299,0.891,0.894,0.003


R² aus Basisvariablen  \
Jahr Variable                                                  
2017 Verfuegbares_Einkommen_EUR_je_EW                   0.75   
     BIP_EUR_je_EW                                      0.93   
2021 Verfuegbares_Einkommen_EUR_je_EW                   0.76   
     BIP_EUR_je_EW                                      0.91   
2025 Verfuegbares_Einkommen_EUR_je_EW                   0.76   
     BIP_EUR_je_EW                                      0.90   

                                                                  stärkste Korrelationen  
Jahr Variable                                                                             
2017 Verfuegbares_Einkommen_EUR_je_EW  Arbeitslosenquote_Pct (-0.65), Schulabgaenger_...  
     BIP_EUR_je_EW                     SV_Beschaeftigte_je_1000_EW (0.91), Auslaender...  
2021 Verfuegbares_Einkommen_EUR_je_EW  Schulabgaenger_ohne_Hauptschulabschluss_Pct (-...  
     BIP_EUR_je_EW                     SV_Beschaeftigte_je_1000_EW (0.91), Alter_60_7...  
2025 Verfuegbares_Einkommen_EUR_je_EW  Arbeitslosenquote_Pct (-0.55), Schulabgaenger_...  
     BIP_EUR_je_EW                     SV_Beschaeftigte_je_1000_EW (0.91), Alter_60_7...

Base model variables: ['Flaeche_km2', 'Bevoelkerung_1000', 'Bevoelkerungsdichte_EW_je_km2', 'Auslaender_Pct', 'Wanderungssaldo_je_1000_EW', 'Alter_unter_18_Pct', 'Alter_18_24_Pct', 'Alter_25_34_Pct', 'Alter_35_59_Pct', 'Alter_60_74_Pct', 'Alter_75plus_Pct', 'Schulabgaenger_ohne_Hauptschulabschluss_Pct', 'Schulabgaenger_Abitur_Pct', 'SV_Beschaeftigte_je_1000_EW', 'Beschaeftigte_Produzierendes_Gewerbe_Pct', 'Arbeitslosenquote_Pct', 'SGBII_Empfaenger_je_1000_EW', 'Ost']


For Report: 

Table 1 shows that the structural variables plus East / West explains 73% - 89% of the variation in AFD voter share between wahlkreise, adding income and GDP only adds 0.003 to it. 

Table 2 explains why. Across the 3 years the Incomes variation between the wahlkreise can be explained up to around 75% by the other variables. For the GDP its even 90%, therefore the columns seem to be redundant and not worth adding the data for 2013 manually. 

### Vehicle Density vs each Party's share

In [7]:
# Does vehicle density (Kfz/Pkw je 1000 EW) correlate with how a Wahlkreis votes?
# Zweitstimmen (party vote) are used, not Erststimmen: Erststimmen mix in candidate effects and tactical
# voting, and parties without a candidate are empty (AfD 2013: 141 of 299 Wahlkreise).
struktur = pd.read_csv("../data/interim/btw_strukturdaten.csv")
wk = pd.read_csv("../data/interim/btw_wahlkreise.csv")

# Zweitstimmen share per party in percent of the valid Zweitstimmen of the Wahlkreis
parties = ["CDU/CSU", "SPD", "Grüne", "FDP", "Linke", "AfD"]
for p in parties:
    wk[p] = wk[f"Zweitstimmen {p}"] / wk["Gültige Zweitstimmen"] * 100

# Join on election year + Wahlkreis number (names differ in dash spelling, numbers do not)
merged = struktur.merge(wk[["Jahr", "Nr"] + parties], left_on=["Jahr", "Wahlkreis_Nr"], right_on=["Jahr", "Nr"])
print(len(merged), "rows (299 Wahlkreise x 4 elections)")

# Kfz (all motor vehicles) exists for 2013/2017, Pkw (cars only) for 2021/2025: one column for both.
# Correlations are computed within each year, so the definition change does not affect them.
merged["Fahrzeuge_je_1000_EW"] = merged["Kfz_je_1000_EW"].fillna(merged["Pkw_je_1000_EW"])

# Pearson correlation across the 299 Wahlkreise of one election: vehicle density vs. each party's share
corr = pd.DataFrame({
    year: group[parties].corrwith(group["Fahrzeuge_je_1000_EW"])
    for year, group in merged.groupby("Jahr")
}).round(2)
corr

1196 rows (299 Wahlkreise x 4 elections)


,2013,2017,2021,2025
CDU/CSU,0.75,0.75,0.63,0.65
SPD,-0.27,-0.13,-0.00,-0.11
Grüne,-0.43,-0.31,-0.50,-0.45
FDP,0.14,-0.02,0.30,0.13
Linke,-0.38,-0.57,-0.63,-0.83
AfD,-0.00,0.07,0.06,0.20


### Fläche_km2 vs each Party's Share

In [8]:
struktur = pd.read_csv("../data/interim/btw_strukturdaten.csv")
wk = pd.read_csv("../data/interim/btw_wahlkreise.csv")

# Zweitstimmen share per party in percent of the valid Zweitstimmen of the Wahlkreis
parties = ["CDU/CSU", "SPD", "Grüne", "FDP", "Linke", "AfD"]
for p in parties:
    wk[p] = wk[f"Zweitstimmen {p}"] / wk["Gültige Zweitstimmen"] * 100

# Join on election year + Wahlkreis number (names differ in dash spelling, numbers do not)
merged = struktur.merge(wk[["Jahr", "Nr"] + parties], left_on=["Jahr", "Wahlkreis_Nr"], right_on=["Jahr", "Nr"])
print(len(merged), "rows (299 Wahlkreise x 4 elections)")

# Pearson correlation across the 299 Wahlkreise of one election: vehicle density vs. each party's share
corr = pd.DataFrame({
    year: group[parties].corrwith(group["Flaeche_km2"])
    for year, group in merged.groupby("Jahr")
}).round(2)
corr

1196 rows (299 Wahlkreise x 4 elections)


,2013,2017,2021,2025
CDU/CSU,0.45,0.41,0.17,0.06
SPD,-0.36,-0.26,-0.02,-0.35
Grüne,-0.55,-0.48,-0.63,-0.60
FDP,-0.37,-0.49,-0.27,-0.37
Linke,0.21,0.04,0.01,-0.31
AfD,0.04,0.36,0.46,0.59


In [18]:
struktur = pd.read_csv("../data/interim/btw_strukturdaten.csv")
wk = pd.read_csv("../data/interim/btw_wahlkreise.csv")

# Zweitstimmen share per party in percent of the valid Zweitstimmen of the Wahlkreis
parties = ["CDU/CSU", "SPD", "Grüne", "FDP", "Linke", "AfD"]
for p in parties:
    wk[p] = wk[f"Zweitstimmen {p}"] / wk["Gültige Zweitstimmen"] * 100

# Join on election year + Wahlkreis number (names differ in dash spelling, numbers do not)
merged = struktur.merge(wk[["Jahr", "Nr"] + parties], left_on=["Jahr", "Wahlkreis_Nr"], right_on=["Jahr", "Nr"])
print(len(merged), "rows (299 Wahlkreise x 4 elections)")

# Pearson correlation across the 299 Wahlkreise of one election: vehicle density vs. each party's share
corr = pd.DataFrame({
    year: group[parties].corrwith(group["Verfuegbares_Einkommen_EUR_je_EW"])
    for year, group in merged.groupby("Jahr")
}).round(2)
corr

1196 rows (299 Wahlkreise x 4 elections)


,2013,2017,2021,2025
CDU/CSU,NaN,0.42,0.42,0.52
SPD,NaN,-0.06,-0.38,-0.13
Grüne,NaN,0.45,0.33,0.40
FDP,NaN,0.64,0.62,0.69
Linke,NaN,-0.60,-0.49,-0.36
AfD,NaN,-0.46,-0.44,-0.41


### Variable Importance (Drop-One)

Which structural variables carry information about the vote that no other variable can replace? This decides which variables stay in the processed dataset.

**Method:** for every election year and party, fit a linear regression (OLS) of the party's Zweitstimmen share on all structural variables and record R² (share of the variation between Wahlkreise that the model explains). Then refit without one variable at a time.

**Score = R²(all variables) − R²(without the variable)**, i.e. the variable's unique contribution.
- Score near 0: the other variables cover its information, so it is redundant.
- Large score: it carries information nothing else has, so it must stay.

**Caveats:**
- Two variables that measure the same thing both score near 0, because each covers for the other when it is dropped. So variables should be removed step by step (drop the weakest, refit, recompute), not all low scores at once.
- The six age shares add up to 100 %, so any one of them is fully determined by the other five and would score 0 by construction. They are scored as one block (all six dropped together).
- 2013 is left out because income and GDP are not published for 2013, so the full model cannot be fitted.
- Kfz (2013/2017) and Pkw (2021/2025) are merged into one vehicle column. Their levels differ by definition, but each model is fitted within one year, so this does not affect the scores.
- R² describes fit, not cause.

In [21]:
# Data: Zweitstimmen share per party, joined with the structural data on year + Wahlkreis number
struktur = pd.read_csv("../data/interim/btw_strukturdaten.csv")
wk = pd.read_csv("../data/interim/btw_wahlkreise.csv")
parties = ["CDU/CSU", "SPD", "Grüne", "FDP", "Linke", "AfD"]
for p in parties:
    wk[p] = wk[f"Zweitstimmen {p}"] / wk["Gültige Zweitstimmen"] * 100
merged = struktur.merge(wk[["Jahr", "Nr"] + parties], left_on=["Jahr", "Wahlkreis_Nr"], right_on=["Jahr", "Nr"])

# East/West flag (five eastern states, Berlin counted as West) and one vehicle column (Kfz 2013/2017, Pkw 2021/2025)
ost_laender = ["Brandenburg", "Mecklenburg-Vorpommern", "Sachsen", "Sachsen-Anhalt", "Thüringen"]
merged["Ost"] = merged["Land"].isin(ost_laender).astype(float)
merged["Fahrzeuge_je_1000_EW"] = merged["Kfz_je_1000_EW"].fillna(merged["Pkw_je_1000_EW"])

# All numeric structural variables (IDs, the two raw vehicle columns and the party shares excluded)
exclude = ["Jahr", "Wahlkreis_Nr", "Nr", "Kfz_je_1000_EW", "Pkw_je_1000_EW"] + parties
variables = [c for c in merged.columns if pd.api.types.is_numeric_dtype(merged[c]) and c not in exclude]

# What gets dropped together: every variable on its own, except the six age shares as one block
age_vars = [v for v in variables if v.startswith("Alter_")]
blocks = {v: [v] for v in variables if v not in age_vars}
blocks["Altersstruktur (6 Anteile)"] = age_vars

In [22]:
def r_squared(data, predictors, target):
    """R² of an OLS fit target ~ predictors (with intercept), via numpy least squares."""
    X = np.column_stack([np.ones(len(data)), data[predictors].values])
    y = data[target].values
    coef, *_ = np.linalg.lstsq(X, y, rcond=None)
    return 1 - (y - X @ coef).var() / y.var()


# For each year (2017-2025) and party: R² with all variables, then the R² loss when each block is dropped
rows = []
for year, g in merged[merged["Jahr"] > 2013].groupby("Jahr"):
    g = g.dropna(subset=variables + parties)  # OLS needs complete rows (2017: 293 of 299 Wahlkreise)
    for party in parties:
        r2_full = r_squared(g, variables, party)
        for name, block in blocks.items():
            others = [v for v in variables if v not in block]
            rows.append({"Jahr": year, "Partei": party, "Variable": name,
                         "R2_full": r2_full, "Score": r2_full - r_squared(g, others, party)})
importance = pd.DataFrame(rows)

#### Results

**Table 1:** score per variable and party, averaged over 2017, 2021 and 2025, sorted by the largest score of any party (`max`).

**Table 2:** R² of the full model per year and party, as a reference for how large the scores are.

In [23]:
overview = importance.pivot_table(index="Variable", columns="Partei", values="Score", aggfunc="mean")[parties]
overview["max"] = overview.max(axis=1)
overview = overview.sort_values("max", ascending=False).round(3)
overview

Partei,CDU/CSU,SPD,Grüne,FDP,Linke,AfD,max
Variable,,,,,,,
Ost,0.038,0.039,0.008,0.004,0.043,0.056,0.056
Altersstruktur (6 Anteile),0.020,0.024,0.043,0.027,0.014,0.029,0.043
Verfuegbares_Einkommen_EUR_je_EW,0.002,0.009,0.007,0.039,0.003,0.002,0.039
SGBII_Empfaenger_je_1000_EW,0.011,0.032,0.000,0.001,0.005,0.001,0.032
Bevoelkerungsdichte_EW_je_km2,0.005,0.006,0.013,0.014,0.030,0.002,0.030
Schulabgaenger_Abitur_Pct,0.004,0.026,0.004,0.005,0.003,0.008,0.026
Auslaender_Pct,0.003,0.002,0.004,0.022,0.004,0.002,0.022
Flaeche_km2,0.005,0.005,0.001,0.017,0.001,0.006,0.017
Beschaeftigte_Produzierendes_Gewerbe_Pct,0.001,0.005,0.016,0.007,0.000,0.010,0.016


In [24]:
importance.groupby(["Jahr", "Partei"])["R2_full"].first().unstack()[parties].round(3)

Partei,CDU/CSU,SPD,Grüne,FDP,Linke,AfD
Jahr,,,,,,
2017,0.758,0.722,0.726,0.641,0.852,0.780
2021,0.692,0.628,0.835,0.612,0.824,0.860
2025,0.821,0.694,0.849,0.647,0.827,0.896


#### Interpretation

- The full model explains 61–90 % of the variation in party shares between Wahlkreise (Table 2).
- **Ost** (up to 0.056, AfD) and **Altersstruktur** (up to 0.043, Grüne) have the largest unique contributions.
- **Income** is redundant for most parties but not for the **FDP** (0.039 on average, 0.03–0.05 per year). The earlier income/GDP check looked only at the AfD; for the FDP, income carries information the other variables do not.
- **Vehicle density** contributes at most 0.011 (SPD), and **GDP** (0.003) and **employees per 1000 inhabitants** (0.004) contribute even less. Their information is covered by the other variables.
- Low scores mean "replaceable as long as the other variables stay", not "can all be removed together". The next step is to drop variables one at a time (weakest first) and recompute after each step.